In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from runner import DualRunner
import pandas as pd

PG_CONNINFO = (
    f"host=127.0.0.1 "
    f"port={os.getenv("POSTGRES_PORT", 5432)} "
    f"dbname={os.getenv("POSTGRES_DB")} "
    f"user={os.getenv("POSTGRES_USER")} "
    f"password={os.getenv("POSTGRES_PASSWORD")}"
)

runner = DualRunner(
    pg_conninfo=PG_CONNINFO,
    duckdb_path=":memory:"
)

display(runner.run_pg("select version()"))
display(runner.run_dd("select version()"))

## pandas 設定

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

,version
0,PostgreSQL 17.7 (Debian 17.7-3.pgdg13+1) on aa...


,"""version""()"
0,v1.4.3


# データ加工のためのSQL
## 一つの値に対する処理

In [3]:
runner.check("""--sql
drop table if exists access_log;
create table access_log (
    stamp timestamp,
    referrer text,
    url text
);
insert into access_log (stamp, referrer, url) values
('2016-08-26 12:02:00', 'http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video/detail?id=001'),
('2016-08-26 12:02:01', 'http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video#ref'),
('2016-08-26 12:02:01', 'https://www.other.com/', 'http://www.example.com/book/detail?id=002');            
select * from access_log;
             
""")

### ✅ SAME

,stamp,referrer,url
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002


In [8]:
runner.pg("""--sql

-- 正規表現を使って値を抽出する
select
    stamp,
    referrer,
    url,
    substring(referrer from 'https?://([^/]*)') as referrer_domain,
    substring(url from '//[^/]+([^?#]+)') as path,
    substring(url from 'id=([^&]*)') as id
from access_log
""")

runner.dd("""--sql

select
    stamp,
    referrer,
    url,
    regexp_extract(referrer, 'https?://([^/]*)', 1) as referrer_domain,
    regexp_extract(url,  '//[^/]+([^?#]+)', 1) as path,
    regexp_extract(url,  'id=([^&]*)', 1) as id
from access_log

""")

### 🐘 PostgreSQL Result

,stamp,referrer,url,referrer_domain,path,id
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001,www.other.com,/video/detail,001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref,www.other.net,/video,None
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002,www.other.com,/book/detail,002


### 🦆 DuckDB Result

,stamp,referrer,url,referrer_domain,path,id
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001,www.other.com,/video/detail,001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref,www.other.net,/video,
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002,www.other.com,/book/detail,002


## 文字列を配列に分解する


